# UnSmile 데이터셋 Fine-tuning (STT 전용 - 자음 제거)

이 노트북은 STT 환경에 맞춰 **자음과 모음(ㅋㅋ, ㅎㅎ 등)을 제거하는 전처리**를 수행한 후 학습을 진행합니다.
데이터 경로: 구글 드라이브 마운트 후 사용

In [ ]:
# [0] 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 작업 디렉토리 설정 (본인의 경로에 맞게 수정하세요)
# os.chdir('/content/drive/MyDrive/SSAFY/UnSmile') 
# print("현재 경로:", os.getcwd())

In [ ]:
# [1] 필수 라이브러리 설치
!pip install -q transformers accelerate emoji soynlp scikit-learn pandas

In [ ]:
# [2] 라이브러리 임포트 및 시드 고정
import os
import re
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

SEED = 42
seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# [3] 설정 및 데이터 경로
# 구글 드라이브 경로 예시 (업로드한 위치로 변경 필요)
TRAIN_FILE = '/content/drive/MyDrive/unsmile_train_v1.0.tsv'
VALID_FILE = '/content/drive/MyDrive/unsmile_valid_v1.0.tsv'

MODEL_NAME = 'beomi/KcELECTRA-base-v2022'
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 2e-5
MAX_LEN = 128

In [ ]:
# [4] 데이터 로드 및 전처리 (STT 대응)

def clean_text(text):
    # 1. 자음/모음만 있는 경우 제거 (예: ㅋㅋ, ㅠㅠ, ㅎㅎ)
    text = re.sub(r'[ㄱ-ㅎㅏ-ㅣ]+', '', str(text))
    # 2. 다중 공백을 하나의 공백으로 줄임
    text = re.sub(r'\s+', ' ', text).strip()
    return text

try:
    train_df = pd.read_csv(TRAIN_FILE, sep='\t')
    val_df = pd.read_csv(VALID_FILE, sep='\t')
    print("데이터 로드 성공!")
    print(f"[Sample Original]: {train_df['문장'].iloc[0]}")

    # 전처리 적용
    train_df['문장'] = train_df['문장'].apply(clean_text)
    val_df['문장'] = val_df['문장'].apply(clean_text)

    # 빈 문장 제거
    train_df = train_df[train_df['문장'].str.len() > 0]
    val_df = val_df[val_df['문장'].str.len() > 0]

    print(f"[Sample Cleaned ]: {train_df['문장'].iloc[0]}")
    print(f"Processed Train size: {len(train_df)}")

except FileNotFoundError:
    print("❌ 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

LABEL_COLUMNS = ['여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭']
num_labels = len(LABEL_COLUMNS)

In [ ]:
# [5] 데이터셋 클래스
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class UnSmileDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['문장'].values
        self.labels = df[LABEL_COLUMNS].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

train_dataset = UnSmileDataset(train_df, tokenizer, MAX_LEN)
val_dataset = UnSmileDataset(val_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# [6] 모델 학습 (전처리 된 데이터로 학습)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_labels, 
    problem_type="multi_label_classification"
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, correct_bias=False)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
loss_fn = torch.nn.BCEWithLogitsLoss()

def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model = model.train()
    losses = []
    for d in tqdm(data_loader, desc="Training"):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, targets)
        losses.append(loss.item())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    return np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    preds = []
    real_targets = []
    with torch.no_grad():
        for d in tqdm(data_loader, desc="Evaluating"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, targets)
            losses.append(loss.item())
            preds.append(torch.sigmoid(outputs.logits).cpu().detach().numpy())
            real_targets.append(targets.cpu().detach().numpy())
    return np.mean(losses), np.vstack(preds), np.vstack(real_targets)

best_f1 = 0
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device, scheduler)
    val_loss, preds, val_targets = eval_model(model, val_loader, loss_fn, device)
    final_preds = (preds > 0.5).astype(int)
    val_f1 = f1_score(val_targets, final_preds, average='macro')
    print(f'Train loss {train_loss:.4f} | Val loss {val_loss:.4f} | Val Macro F1 {val_f1:.4f}')
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')

In [ ]:
# [9] 자음 없는 입력 테스트
def predict_sentence(sentence):
    sentence = clean_text(sentence) # 예측 시에도 동일 전처리 적용
    model.eval()
    encoding = tokenizer.encode_plus(
        sentence,
        add_special_tokens=True,
        max_length=MAX_LEN,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = encoding['input_ids'].to(device).flatten().unsqueeze(0)
    attention_mask = encoding['attention_mask'].to(device).flatten().unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    print(f"문장: {sentence}")
    for label, prob in zip(LABEL_COLUMNS, probs):
        if prob > 0.5:
            print(f" - {label}: {prob*100:.2f}%")

predict_sentence("이런 쓰레기 같은 영화는 처음 본다 ㅋㅋㅋ") # 자음 제거되어 테스트됨
predict_sentence("정말 유익하고 좋은 정보 감사합니다")